In [2]:
import json
import os
import sys

In [3]:
from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam, ChatCompletionToolParam

In [4]:
api_key = os.getenv("OPENAI_API_KEY")

In [6]:
client = OpenAI(
        base_url="https://api.deepseek.com/v1",
        api_key=api_key
    )

In [7]:
def get_current_weather(city: str) -> dict:
    fake_db = {"Paris": "18°C, light rain", "Tokyo": "27°C, sunny"}
    return {"city": city, "conditions": fake_db.get(city, "unknown")}

In [8]:
get_current_weather("Paris")


{'city': 'Paris', 'conditions': '18°C, light rain'}

In [9]:
tools: list[ChatCompletionToolParam] = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, e.g. Paris"},
                },
                "required": ["city"],
                "additionalProperties": False,
            },
        },
    }
]

In [10]:
tools


[{'type': 'function',
  'function': {'name': 'get_current_weather',
   'description': 'Get the current weather for a city.',
   'parameters': {'type': 'object',
    'properties': {'city': {'type': 'string',
      'description': 'City name, e.g. Paris'}},
    'required': ['city'],
    'additionalProperties': False}}}]

In [11]:
messages: list[ChatCompletionMessageParam] = [
    {"role": "user", "content": "What's the weather like in Tokyo?"}
]

In [12]:
first = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=messages,
    tools=tools,
)

In [17]:
reply = first.choices[0].message

In [18]:
print(reply)

ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_VO1abADaaAwOfA3ghOmx2740', function=Function(arguments='{"city": "Tokyo"}', name='get_current_weather'), type='function', index=0)], reasoning_content="The user asks about the weather in Tokyo. I'll call the weather tool.")


In [19]:
messages.append(reply)

In [20]:
messages

[{'role': 'user', 'content': "What's the weather like in Tokyo?"},
 ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_VO1abADaaAwOfA3ghOmx2740', function=Function(arguments='{"city": "Tokyo"}', name='get_current_weather'), type='function', index=0)], reasoning_content="The user asks about the weather in Tokyo. I'll call the weather tool.")]

In [22]:
args = json.loads(reply.tool_calls[0].function.arguments)

In [23]:
args

{'city': 'Tokyo'}

In [24]:
print(f"[model requested: {reply.tool_calls[0].function.name}({args})]")

[model requested: get_current_weather({'city': 'Tokyo'})]


In [25]:
result = get_current_weather(**args)

In [26]:
result


{'city': 'Tokyo', 'conditions': '27°C, sunny'}

In [27]:
messages.append({
        "role": "tool",
        "tool_call_id": reply.tool_calls[0].id,
        "content": json.dumps(result),
    })

In [28]:
messages

[{'role': 'user', 'content': "What's the weather like in Tokyo?"},
 ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_VO1abADaaAwOfA3ghOmx2740', function=Function(arguments='{"city": "Tokyo"}', name='get_current_weather'), type='function', index=0)], reasoning_content="The user asks about the weather in Tokyo. I'll call the weather tool."),
 {'role': 'tool',
  'tool_call_id': 'call_00_VO1abADaaAwOfA3ghOmx2740',
  'content': '{"city": "Tokyo", "conditions": "27\\u00b0C, sunny"}'}]

In [29]:
second = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=messages,
    tools=tools,
)

In [30]:
print("\n" + (second.choices[0].message.content or ""))


The weather in Tokyo is currently **27°C and sunny**! ☀️


In [32]:
print(second.choices[0])



Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The weather in Tokyo is currently **27°C and sunny**! ☀️', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content=''))
